In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set working directory to project root so all relative paths work consistently.
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("02_eda.ipynb"))))
print(f"Working directory: {os.getcwd()}")

# ── Load and label ────────────────────────────────────────────────────────
# Load the two 5-second aggregation window CSV files from the UNB IIoT Dataset 2025.
# We chose the 5sec window as a balance between temporal resolution and feature stability.
# Shorter windows (1-2sec) are too noisy; longer windows (8-10sec) average out attack signatures.
attack = pd.read_csv("data/raw/attack_samples_5sec.csv")
benign = pd.read_csv("data/raw/benign_samples_5sec.csv")

# Assign binary labels: 1 = attack, 0 = benign.
# We create our own label column rather than using the original label columns to ensure a clean binary target with no ambiguity.
attack["label"] = 1
benign["label"] = 0

# Combine into a single DataFrame and shuffle with a fixed random seed
# for reproducibility. Shuffling prevents any ordering bias during training
df = pd.concat([attack, benign], ignore_index=True).sample(frac=1, random_state=42)

print(f"Combined shape: {df.shape}")
print(f"\nClass distribution:\n{df['label'].value_counts()}")
print(f"\nAttack ratio: {df['label'].mean():.2%}")

In [ ]:
# ── Class distribution plot ───────────────────────────────────────────────
# Visualize class balance to understand the imbalance between attack and benign samples.
# A 39/61 split is a moderate imbalance — not severe enough to require aggressive
# resampling (SMOTE), but enough that we will use class weighting during training.
# Accuracy alone will be misleading here; F1 (penalizes the model heavily for missing attacks entirely)
# and AUC-ROC (measures how well the model separates attack from benign across all possible thresholds) are our primary metrics.
os.makedirs("results", exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
counts = df['label'].value_counts()
bars = ax.bar(['Benign (0)', 'Attack (1)'], counts.values,
               color=['#0d9488', '#7c3aed'], width=0.5)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count:,}', ha='center', fontsize=11)

ax.set_title("Class Distribution", fontsize=13)
ax.set_ylabel("Sample Count")
ax.set_ylim(0, max(counts.values) * 1.15)
plt.tight_layout()
plt.savefig("results/class_distribution.png", dpi=150)
plt.show()
print("Saved to results/class_distribution.png")

In [ ]:
# ── Drop identifier and unused label columns ──────────────────────────────
# Drop columns that are identifiers, timestamps, and redundant label columns.
# These are removed for two key reasons:
# 1. Identifiers (IPs, MACs, device names) are unique per session/device.
#    If kept, the model would memorize "this IP = attack" rather than learning generalizable traffic behavior patterns — causing inflated   ` results that won't hold on unseen data.
# 2. Timestamps are removed to prevent temporal leakage — the model should learn from traffic statistics, not when the traffic occurred.
# 3. Original label columns (label_full, label1-4) are dropped because we created our own clean binary label column above. Keeping them       risks data leakage if any accidentally remain in the feature set.
drop_cols = [
    'device_name', 'device_mac',
    'network_ips_all', 'network_ips_src', 'network_ips_dst',
    'network_macs_all', 'network_macs_src', 'network_macs_dst',
    'network_ports_all', 'network_ports_src', 'network_ports_dst',
    'network_protocols_all', 'network_protocols_src', 'network_protocols_dst',
    'timestamp', 'timestamp_start', 'timestamp_end',
    'label_full', 'label1', 'label2', 'label3', 'label4'
]

df.drop(columns=drop_cols, inplace=True)

print(f"Shape after dropping identifiers: {df.shape}")
print(f"\nRemaining columns:\n{list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes.value_counts()}")

In [ ]:
# ── Find remaining string column ──────────────────────────────────────────
# Check for any remaining non-numeric columns that would break model training.
# Deep learning models require all inputs to be numeric — any string columns must either be encoded or dropped depending on their information value.
str_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Remaining string columns: {str_cols}")

for col in str_cols:
    print(f"\n{col} — unique values: {df[col].nunique()}")
    print(df[col].value_counts().head(10))

In [ ]:
# ── Drop log_data-types string column ────────────────────────────────────
# Drop log_data-types — this column stores lists of data types as strings
# (e.g. ['numeric'], ['array', 'numeric']). Two reasons to drop:
# 1. 73% of values are empty ([]) — a feature that's blank for most samples adds noise rather than signal.
# 2. The remaining values describe log-level metadata, not network traffic behavior. It won't help the model distinguish attacks from benign traffic.
df.drop(columns=['log_data-types'], inplace=True)

print(f"Shape after dropping log_data-types: {df.shape}")
print(f"Dtypes remaining:\n{df.dtypes.value_counts()}")

In [ ]:
# ── Check for constant columns ────────────────────────────────────────────
# Identify highly correlated feature pairs (correlation > 0.95).
# Highly correlated features carry nearly identical information — keeping both adds redundancy without improving model performance, and can slow training.
# We use the upper triangle of the correlation matrix to avoid counting each pair twice (A-B and B-A).
feature_cols = [c for c in df.columns if c != 'label']
const_cols = [c for c in feature_cols if df[c].nunique() <= 1]
print(f"Constant columns (drop these): {const_cols}")

# ── Check for highly correlated features ──────────────────────────────────
corr = df[feature_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [(col, row, upper.loc[row, col])
             for col in upper.columns
             for row in upper.index
             if upper.loc[row, col] > 0.95]

print(f"\nHighly correlated pairs (>0.95): {len(high_corr)}")
for col1, col2, val in high_corr[:15]:
    print(f"  {col1} — {col2}: {val:.3f}")

In [ ]:
# ── Drop highly correlated features ──────────────────────────────────────
# Drop one column from each highly correlated pair.
# Strategy: for each pair above the 0.95 threshold, we add the row variable to the drop set. This keeps the column variable (which may appear in multiple pairs) and removes its redundant counterparts.
# This reduces 71 features to 50 — removing 21 redundant columns while preserving the unique information content of the dataset.
cols_to_drop = set()
for col in upper.columns:
    for row in upper.index:
        if upper.loc[row, col] > 0.95:
            cols_to_drop.add(row)

print(f"Columns to drop: {len(cols_to_drop)}")
print(cols_to_drop)

df.drop(columns=list(cols_to_drop), inplace=True)
print(f"\nShape after dropping correlated features: {df.shape}")

In [ ]:
# ── Feature distributions ─────────────────────────────────────────────────
# Plot distributions of all 50 remaining features.
# Most features are expected to be right-skewed (values clustered near zero with long tails) — this is typical for network traffic statistics.
# Key observations:
# - Heavy skew confirms we need StandardScaler in preprocessing (Day 3)
# - Features showing two distinct spikes suggest separation between benign and attack behavior — a positive signal for model performance
# - No flat/constant distributions confirm our correlation drop was effective
feature_cols = [c for c in df.columns if c != 'label']

fig, axes = plt.subplots(10, 5, figsize=(20, 30))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=40, color='#0d9488', alpha=0.7)
    axes[i].set_title(col, fontsize=7)
    axes[i].set_yticks([])

# Hide any unused subplots
for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions (50 features)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("results/feature_distributions.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to results/feature_distributions.png")

In [9]:
# ── Save cleaned dataframe for Day 3 preprocessing ───────────────────────
# Save the cleaned, EDA-processed DataFrame to disk for use in Day 3 preprocessing.
# This file contains:
# - 45,055 samples (17,695 attack + 27,360 benign)
# - 50 numeric features (all identifiers, string columns, and redundant
#   correlated features have been removed)
# - 1 binary label column (1=attack, 0=benign)
# - 0 missing values
# This file is the starting point for normalization and train/val/test splitting.
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/cleaned_eda.csv", index=False)

print(f"Final shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Samples: {df.shape[0]}")
print(f"Saved to data/processed/cleaned_eda.csv")

Final shape: (45055, 51)
Features: 50
Samples: 45055
Saved to data/processed/cleaned_eda.csv
